# DAN objective
Written only; no cells executed. See task2/README.md for execution order.

In [ ]:
def multi_kernel_mmd(source, target, scales=(0.5, 1.0, 2.0)):
    """Biased empirical squared RKHS distance, including diagonal kernel terms.

    k(a,b) = sum_s exp(-||a-b||^2 / (s * median_distance_squared)).
    Median uses all unique off-diagonal pairs in the current combined batch.
    Bandwidth estimation is detached; gradients flow through both feature sets.
    Kernels are summed, not averaged. No feature normalization is applied.
    """
    if source.ndim != 2 or target.ndim != 2 or source.shape[1] != target.shape[1]:
        raise ValueError('Expected two nonempty feature matrices with matching width.')
    if not len(source) or not len(target):
        raise ValueError('MMD needs source and target examples.')
    if not scales or any(s <= 0 for s in scales):
        raise ValueError('Kernel scales must be positive.')
    features = torch.cat((source, target), dim=0).float()
    norm = features.square().sum(dim=1, keepdim=True)
    distances = (norm + norm.T - 2 * features @ features.T).clamp_min(0)
    # Enforce the exact self-distance rather than retaining roundoff on the diagonal.
    distances = distances - torch.diag_embed(distances.diagonal())
    pairs = torch.triu_indices(len(features), len(features), offset=1, device=features.device)
    median = distances.detach()[pairs[0], pairs[1]].median().clamp_min(1e-8)
    kernel = sum(torch.exp(-distances / (float(scale) * median)) for scale in scales)
    n = len(source)
    return kernel[:n, :n].mean() + kernel[n:, n:].mean() - 2 * kernel[:n, n:].mean()


def dan_objective(model, x, y, target_x, discriminator, cfg, progress, rng_state):
    source_features = forward_features(model, x)
    target_features = forward_features(model, target_x)
    logits = model.fc(source_features)
    classification = nn.functional.cross_entropy(logits, y)
    alignment = multi_kernel_mmd(source_features, target_features, cfg['kernel_scales'])
    loss = classification + cfg['lambda_mmd'] * alignment
    return logits, loss, classification, alignment, 0, 0, 0.0, rng_state